In [ ]:
# Colab bootstrap: run this cell first
REPO_URL = "https://github.com/RolexMaster/Phys.AIAgent.git"
PROJECT_ROOT = "/content/Phys.AIAgent"

!rm -rf {PROJECT_ROOT}
!git clone {REPO_URL} {PROJECT_ROOT}

import os
os.environ["PROJECT_ROOT"] = PROJECT_ROOT

!find {PROJECT_ROOT} -maxdepth 2 -type d \( -name src -o -name scenarios -o -name notebooks \)


In [ ]:
from pathlib import Path
import os
import sys

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError as exc:
    if exc.name != "imp":
        raise
    print("autoreload is unavailable in this Python 3.12 environment because IPython still imports imp.")
except Exception as exc:
    print(f"autoreload is unavailable: {exc}")

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", "/content/Phys.AIAgent")).expanduser().resolve()
SRC_DIR = PROJECT_ROOT / "src"
SCENARIO_DIR = PROJECT_ROOT / "scenarios"
SERVER_LOG = PROJECT_ROOT / "server.log"

if not SRC_DIR.exists():
    raise RuntimeError(f"src not found: {SRC_DIR}")
if not SCENARIO_DIR.exists():
    raise RuntimeError(f"scenarios not found: {SCENARIO_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

DRIVE_ROOT = None
DRIVE_MODELS_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive").resolve()
    DRIVE_MODELS_DIR = DRIVE_ROOT / "hf-models"
    DRIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted: {DRIVE_MODELS_DIR}")
except Exception as exc:
    print(f"Google Drive mount skipped: {exc}")

HF_TOKEN = os.getenv("HF_TOKEN", "").strip()

try:
    from huggingface_hub import whoami
    if HF_TOKEN:
        _hf_user = whoami(token=HF_TOKEN)
        print(f"HF token verified for: {_hf_user.get('name', 'unknown')}")
    else:
        print("HF_TOKEN is not set. Public model downloads only.")
except Exception as exc:
    print(f"HF token check failed: {exc}")
    print("Public model downloads will retry without auth if possible.")

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SRC_DIR: {SRC_DIR}")
print(f"SCENARIO_DIR: {SCENARIO_DIR}")
if DRIVE_MODELS_DIR is not None:
    print(f"DRIVE_MODELS_DIR: {DRIVE_MODELS_DIR}")
print("Environment setup complete")


In [ ]:
import os
import socket
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

MCP_URL = os.getenv("MCP_URL", "https://page-romantic-webpage-terrace.trycloudflare.com/mcp").strip()
os.environ["MCP_URL"] = MCP_URL

parsed = urlparse(MCP_URL)
if not parsed.scheme or not parsed.netloc:
    raise RuntimeError(f"Invalid MCP_URL: {MCP_URL}")

host = parsed.hostname
port = parsed.port or (443 if parsed.scheme == "https" else 80)
print(f"MCP_URL: {MCP_URL}")
print(f"Resolving MCP host: {host}:{port}")
resolved = socket.getaddrinfo(host, port, type=socket.SOCK_STREAM)
resolved_ips = sorted({item[4][0] for item in resolved})
print("DNS OK:", ", ".join(resolved_ips[:3]))

request = Request(MCP_URL, method="GET", headers={"Accept": "application/json, text/event-stream"})
try:
    with urlopen(request, timeout=10) as response:
        status = response.status
        print(f"HTTP GET OK: {status}")
except HTTPError as exc:
    status = exc.code
    print(f"HTTP GET responded: {status}")
except URLError as exc:
    raise RuntimeError(f"MCP endpoint is unreachable: {exc}") from exc

if status >= 500:
    raise RuntimeError(f"MCP endpoint responded with server error: HTTP {status}")

print("MCP endpoint reachability check passed")


In [ ]:
import torch, platform
print("CUDA available:", torch.cuda.is_available())
print("torch:", torch.__version__)
print("python:", platform.python_version())

# vLLM ? ??? ????? ?? ?? ????
print("vLLM and Torch reinstall starts...")
!pip uninstall -y vllm torch torchvision
!pip cache purge
!pip install torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121
!pip -q install -U "vllm>=0.8.5" "transformers>=4.51.0" "huggingface_hub>=0.23.0" openai

print("Hugging Face login already verified")


In [ ]:
# vllm만 다시 설치
!pip install -U "vllm>=0.8.5"

import torch, vllm
print("torch:", torch.__version__)
print("vllm :", vllm.__version__)
print("CUDA :", torch.version.cuda)

!pip install fastmcp


In [ ]:
import subprocess

import torch

GPU_REQUIRED = True
GPU_AVAILABLE = torch.cuda.is_available()
GPU_DEVICE_COUNT = torch.cuda.device_count()

print("CUDA available:", GPU_AVAILABLE)
print("CUDA device count:", GPU_DEVICE_COUNT)
print("Torch CUDA:", torch.version.cuda)

nvidia_smi_result = None
try:
    nvidia_smi_result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if nvidia_smi_result.stdout:
        print(nvidia_smi_result.stdout)
    if nvidia_smi_result.stderr:
        print(nvidia_smi_result.stderr)
except FileNotFoundError:
    print("nvidia-smi is not available in this runtime.")

if GPU_REQUIRED and (
    not GPU_AVAILABLE
    or GPU_DEVICE_COUNT < 1
    or nvidia_smi_result is None
    or nvidia_smi_result.returncode != 0
):
    raise RuntimeError(
        "No CUDA GPU runtime detected. This notebook uses vLLM and will not start without a GPU. "
        "In Colab, switch to a GPU runtime, restart the session, and run the notebook again."
    )

!df -h /dev/shm
!free -h


In [ ]:
import os
import shutil
from pathlib import Path

from huggingface_hub import snapshot_download
from phys_ai_agent.config import resolve_model_config, resolve_runtime_config
from phys_ai_agent.vllm import download_model

runtime_config = resolve_runtime_config(project_root=PROJECT_ROOT)
MODEL_FAMILY = runtime_config.model_family
model_cfg = resolve_model_config(MODEL_FAMILY)
MODEL_REPO_ID = runtime_config.model_repo_id
MODEL_LOCAL_DIR = runtime_config.model_local_dir
MODEL_LOCAL_PATH = Path(MODEL_LOCAL_DIR)
SERVED_MODEL_NAME = runtime_config.served_model_name
DRIVE_MODEL_DIR = DRIVE_MODELS_DIR / MODEL_LOCAL_PATH.name if DRIVE_MODELS_DIR else None

os.environ["MODEL_FAMILY"] = MODEL_FAMILY
os.environ["LLM_MODEL"] = runtime_config.llm_model
os.environ["LLM_BASE_URL"] = runtime_config.llm_base_url

def _has_model_files(path: Path) -> bool:
    return path.exists() and (path / "config.json").exists()

if _has_model_files(MODEL_LOCAL_PATH):
    model_path = str(MODEL_LOCAL_PATH)
    print(f"Using existing local model cache: {model_path}")
elif DRIVE_MODEL_DIR and _has_model_files(DRIVE_MODEL_DIR):
    print(f"Copying model from Google Drive cache: {DRIVE_MODEL_DIR} -> {MODEL_LOCAL_PATH}")
    MODEL_LOCAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_MODEL_DIR, MODEL_LOCAL_PATH, dirs_exist_ok=True)
    model_path = str(MODEL_LOCAL_PATH)
else:
    try:
        model_path = download_model(model_cfg, hf_token=HF_TOKEN)
    except Exception as exc:
        _msg = str(exc).lower()
        if "401" in _msg or "unauthorized" in _msg or "expired" in _msg:
            print(f"HF auth failed, retrying public download without token: {exc}")
            os.environ.pop("HF_TOKEN", None)
            try:
                from huggingface_hub import logout
                logout()
            except Exception:
                pass
            model_path = snapshot_download(repo_id=model_cfg.repo_id, local_dir=model_cfg.local_dir)
        else:
            raise

if DRIVE_MODEL_DIR and _has_model_files(MODEL_LOCAL_PATH) and not _has_model_files(DRIVE_MODEL_DIR):
    print(f"Syncing model to Google Drive cache: {MODEL_LOCAL_PATH} -> {DRIVE_MODEL_DIR}")
    DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(MODEL_LOCAL_PATH, DRIVE_MODEL_DIR, dirs_exist_ok=True)
    print(f"Google Drive cache updated: {DRIVE_MODEL_DIR}")

print(f"Selected model family: {MODEL_FAMILY}")
print("Model path:", model_path)
if DRIVE_MODEL_DIR:
    print("Drive cache path:", DRIVE_MODEL_DIR)


In [ ]:
from phys_ai_agent.vllm import build_vllm_server_command

VLLM_HOST = "127.0.0.1"
VLLM_PORT = 8000
VLLM_BASE_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
VLLM_STARTUP_LOG_DELAY_SEC = 10

if not globals().get("GPU_AVAILABLE", False) or globals().get("GPU_DEVICE_COUNT", 0) < 1:
    raise RuntimeError(
        "GPU preflight did not pass. Refusing to start vLLM without a CUDA GPU."
    )

server_command = build_vllm_server_command(model_cfg, port=VLLM_PORT, gpu_memory_utilization=0.80)
print("Server command:", server_command)
print("Served model:", model_cfg.served_model_name)
print("LLM base URL:", VLLM_BASE_URL)
print("Models endpoint:", f"{VLLM_BASE_URL}/models")
print("Chat endpoint:", f"{VLLM_BASE_URL}/chat/completions")

!pkill -f "vllm.entrypoints.openai.api_server" || true
!nohup {server_command} > {SERVER_LOG} 2>&1 &
!sleep {VLLM_STARTUP_LOG_DELAY_SEC}; tail -n 80 {SERVER_LOG}


In [ ]:
import json
import time
from pathlib import Path
from urllib.request import urlopen

VLLM_READY_MAX_ATTEMPTS = 60
VLLM_READY_POLL_INTERVAL_SEC = 5
VLLM_READY_REQUEST_TIMEOUT_SEC = 15

models_url = f"{VLLM_BASE_URL}/models"
last_error = None

print(
    "Waiting for vLLM readiness:",
    f"attempts={VLLM_READY_MAX_ATTEMPTS}, poll_interval={VLLM_READY_POLL_INTERVAL_SEC}s, ",
    f"request_timeout={VLLM_READY_REQUEST_TIMEOUT_SEC}s",
)

for attempt in range(1, VLLM_READY_MAX_ATTEMPTS + 1):
    try:
        with urlopen(models_url, timeout=VLLM_READY_REQUEST_TIMEOUT_SEC) as response:
            payload = json.loads(response.read().decode("utf-8"))
        model_ids = [item.get("id") for item in payload.get("data", [])]
        print(f"[LLM READY] attempt {attempt}: status=OK")
        print("served models:", model_ids)
        break
    except Exception as exc:
        last_error = exc
        print(f"[LLM WAIT] attempt {attempt}/{VLLM_READY_MAX_ATTEMPTS}: {type(exc).__name__}: {exc}")
        if attempt < VLLM_READY_MAX_ATTEMPTS:
            time.sleep(VLLM_READY_POLL_INTERVAL_SEC)
else:
    print(f"[LLM ERROR] check server log: {SERVER_LOG}")
    server_log_path = Path(SERVER_LOG)
    if server_log_path.exists():
        print("[LLM ERROR] last 80 lines from server.log:")
        log_lines = server_log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        for line in log_lines[-80:]:
            print(line)
    total_wait_sec = (VLLM_READY_MAX_ATTEMPTS - 1) * VLLM_READY_POLL_INTERVAL_SEC
    raise RuntimeError(
        f"vLLM server is not ready after {VLLM_READY_MAX_ATTEMPTS} attempts "
        f"(~{total_wait_sec}s wait): {last_error}"
    )


In [ ]:
# ?? 50?? ??? ?, ??? ????(??? ??) ??
!tail -n 50 {SERVER_LOG} | tac


In [ ]:
from phys_ai_agent import bridge_session

print(bridge_session.__version__)
print(bridge_session.__commit__)
print(bridge_session.BridgeSession.VERSION, bridge_session.BridgeSession.COMMIT)

In [ ]:
!python -m py_compile {SRC_DIR / 'phys_ai_agent' / 'bridge_session.py'} {SRC_DIR / 'phys_ai_agent' / 'agent_prompts.py'}


In [ ]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

from phys_ai_agent.config import resolve_runtime_config
from phys_ai_agent.runner import configure_file_logger, run_preset_queries

runtime_config = resolve_runtime_config(project_root=PROJECT_ROOT)

print(f"Selected model family: {runtime_config.model_family}")
print(f"LLM_MODEL: {runtime_config.llm_model}")
print(f"MCP_URL: {runtime_config.mcp_url}")
print(f"Scenario file: {runtime_config.scenario_file}")
print(f"Preset query count: {len(runtime_config.preset_queries)}")

transport = StreamableHttpTransport(url=runtime_config.mcp_url)
async with Client(transport) as mcp:
    await mcp.ping()
    tools = await mcp.list_tools()

print(f"MCP ping OK. tool count: {len(tools)}")

logger = configure_file_logger(runtime_config.log_file)
results = await run_preset_queries(
    runtime_config,
    logger=logger,
    session_max_turns=runtime_config.max_turns,
)


In [ ]:
!tail -n 50 {runtime_config.log_file}


In [ ]:
from phys_ai_agent.runner import smoke_test_chat_completion

response_text = smoke_test_chat_completion(
    base_url=runtime_config.llm_base_url,
    model=runtime_config.llm_model,
    api_key=runtime_config.llm_api_key,
    message="Colab?? vLLM ?? ??????",
)
print(response_text)
